**2D**
1) Define the domain $\Omega$
2) Generate triangulations T_h
3) Construct finite element space V_h
4) Get local stiffness matrix A_k
5) Get local mass matrix M_k
6) Assemble local stiffness matrices into global stiffness matrix A
7) Assemble local mass matrices into a global mass matrix M
8) Impose Dirichlet Boundary Condition
9) Solve $A * u = \lambda * M * u$
10) Extract eigenvalues
11) compute the eigenvectors
12) Compare with exact solution


**2D FEM Implementation notes**
1. Generated mesh using Delaunay triangulation
2. Computed local element matrices
3. Assembled global stiffness and mass matrices
4. Identified boundaru and interior nodes
5. Imposed homogeneous Dirichlet boundary conditions by removing rows and columns associated with boundary nodes
6. Solved reduced generalised eigenvalue problem

*For future*
- global matrices are sparse (majority of elements are 0) and symmetric
- boundary conditions reduce the dimention of the system
- enumerate shows which node number corresponds to the coordinates
- with 1 interior node we have 1 degree of freedom, the eigenvalue was 48, comparing to the exact first Dirichlet eigenvalue on the unit square is $\lambda_1 = 2 * \pi^2 \approx 19.739$
- the exact eigenvalue for the unit square with Dirichlet BC is $\lambda_m,n = \pi^2 (m^2 + n^2)$
- with a finer mesh, where there are 5 points on the axis the eigenvalue was 22.506 which is much closer to the real one 19.739
- for generalised FEM eigenproblem the common normalisation is $u^T M u = 1$
- shape is more important when printing the eigenvector (it is 0 on the boundary and maximum in the centre), which is the discrete approximation of the first eigenfunction of the unit square $u(x, y) = sin(\pi x) * sin(\pi y)$. If we take the finer mesh the numbers will be the coefficients of the $u_h = \sum_i u_i \phi_i (x)$. 

**The vector tells us how much of each basis function is present in the final FEM approximation**.
*The numbers in the eigenvector are coefficients at the interior nodes, not values at every point of the domain.*

**TO DO**
1) Compute:  $P_e v_k = ⟨v_k, u_1,2⟩ * u_1,2 + ⟨v_k, u_2,1⟩ * u_2,1$
Then check  ‖vₖ − Pₑ vₖ‖  converges to 0 as h → 0. Where P_e projects onto the span ${u_1,2 , u_2,1}$
2) make not the L2 norm of the error of the eigenfunctions, make the $H^1$ norm to do so follow the formula $\left\lVert u - u_h \right\rVert _{H^1} = (\int_{\Omega}(u - u_h)^2 + |\nabla (u - u_h)|^2 dx)^{1/2}$
- compute centroid
x_c = np.mean(x_coords)
y_c = np.mean(y_coords)
- evaluate gradient there
grad_u_exact = np.array([np.pi*np.cos(np.pi*x_c)*np.sin(np.pi*y_c),
       np.pi*np.sin(np.pi*x_c)*np.cos(np.pi*y_c)
])
- compute fem gradient
grad_u_h = (
      u_local[0]*grad_phi[0]
    + u_local[1]*grad_phi[1]
    + u_local[2]*grad_phi[2]
)
- accumulate 
H1_error_sq += T_k * np.dot(grad_u_exact - grad_u_h, grad_u_exact - grad_u_h)
- find error
H1_error = np.sqrt(H1_error_sq)

In [ ]:
# 2D let the domain be uniform and defined on [0, 1]x[0, 1]
import numpy as np
from scipy.linalg import eigh
from scipy.spatial import Delaunay
import pandas as pd
import math
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [ ]:
### FEM METHODS ###

def generate_mesh(n_points: int):

       '''
       Generates a 2D mesh over the unit square [0, 1] x [0, 1]

       Places n_points evenly along each axis, builds coordinate grid, 
              and divides into triangulations using Delaunay triangulation
       
       :param n_points: a number of points along each axis
       :return: returns domain - an (N, 2) array of 2D domain coordinates 
              and tri - nodes of the trinagle in the global domain
       '''

       # define the axis intervals
       x = np.linspace(0, 1, n_points)
       y = np.linspace(0, 1, n_points)

       # coordinate generation via meshgrid (takes 1D arrays and duplicates them to build 2D grids)
       X, Y = np.meshgrid(x, y)

       # c_ matches the first X with the first Y, second X with second Y etc
       # ravel() takes 2D matrix structure and reads it row by row into a long single list of coordinates
       # Delaunay cannot read 2D grid, thats why we flatten X and Y, so they can be paired together
       domain = np.c_[X.ravel(), Y.ravel()]
       

       # create triangles on the domain
       tri = Delaunay(domain)

       return domain, tri



def stiffness_matrix_A(grad_phi):

       ''' 
       stiffness_matrix_A creates a local 3x3 stiffness matrix

       :param grad_phi: a numpy array of the gradients of the base functions of the triangle
       :return: returns a stiffness matrix A
       '''

       # start with matrix of zeros
       A_lower_tri = np.zeros((3, 3), dtype=float)
       # off diagonal values
       # first make lower triangular matrix
       # A_2_1
       A_lower_tri[1, 0] = np.dot(grad_phi[1], grad_phi[0])
       # A_3_1
       A_lower_tri[2, 0] = np.dot(grad_phi[2], grad_phi[0])
       # A_3_2
       A_lower_tri[2, 1] = np.dot(grad_phi[2], grad_phi[1])

       A = create_symmetric_matrix(A_lower_tri)
       # diagonal values
       A[0, 0] = np.dot(grad_phi[0], grad_phi[0])
       A[1, 1] = np.dot(grad_phi[1], grad_phi[1])
       A[2, 2] = np.dot(grad_phi[2], grad_phi[2])
       
       return A

def create_symmetric_matrix(lower_tri_matrix):

       '''
       This method creates a symmetric matrix out of lower triangulated matrix 
              by adding it with its transpose
       
       :param lower_tri_matrix: a lower triangular matrix
       :return: returns a symmetric matrix
       '''

       # Zero the diagonal so it isnt double counted, when added
       lower_tri_no_diag = lower_tri_matrix.copy()
       np.fill_diagonal(lower_tri_no_diag, 0)
       
       # create symmetric matrix by adding the lower triangular matrix to its transpose
       sym_matrix = lower_tri_no_diag + lower_tri_no_diag.T

       return sym_matrix


def mass_matrix_M():

       ''' 
       mass_matrix_M stores the local mass matrix, because it is the same for every mesh, 
              the only difference would be the scalling by the area of the triangle, 
              which is handled by triangle_solver
       
       :return: returns a local mass matrix
       '''
       
       return np.array(
              [
                     [2, 1, 1],
                     [1, 2, 1],
                     [1, 1, 2]
              ]
       )



def local_triangle_matrices(coords_of_triangle):

       '''
       local_triangle_matrices receives the coordinates of the triangle in the domain and calculates:
              the stiffness matrix - measures the allignment of the gradients of each basis function on this triangle
              the mass matrix - measures the overlap of the heights of each basis function on this triangle
       
       :param coords_of_triangle: the (N, 2) numpy array of the global coordintes of the triangle
       :return: returns the computed local stiffness and mass matrices
       '''

       # find the area of the triangle
       col = np.array([1, 1, 1])
       # create 3x3 matrix to find the area
       coords_matrix = np.hstack((coords_of_triangle, np.atleast_2d(col).T))

       # area of a triangle
       T_k = 0.5 * abs(np.linalg.det(coords_matrix))

       x = []
       y = []
       # for coordinate in all of the coordinates of the nodes of this triangle
       for coord in coords_of_triangle:
              x.append(float(coord[0]))
              y.append(float(coord[1]))

       # basis function has an equation phi = a + bx + cy, where b and c are the coefficients of the coordinates
       # so we find b and c for every basis function of the triangle
       c = []   
       c.append(x[2] - x[1])
       c.append(x[0] - x[2])
       c.append(x[1] - x[0])

       b = []
       b.append(y[1] - y[2])
       b.append(y[2] - y[0])
       b.append(y[0] - y[1])

       # find the gradients of the basis functions
       grad_phi = np.array([
              (1 / (2 * T_k)) * np.array([b_i, c_i]) for b_i, c_i in zip(b, c)
       ])


       # get local stiffness matrix
       A_local = T_k * stiffness_matrix_A(grad_phi)
       
       # get local mass matrix
       M_local = (T_k / 12) * mass_matrix_M()
       
       return A_local, M_local

# put the triangle in the global matrix
def put_local_to_global(global_matrix, local_matrix, coord):

       '''
       This method puts local matrices into the corresponding global ones according to their position in the domain

       :param global_matrix: the global matrix (the matrix of the domain) either empty if the first node or already have previous local matrices in it
       :param local_matrix: the matrix of the triangle
       :param coord: the coordinates of the triangle in the domain

       :return: return the obtained global matrix
       '''

       n_local = local_matrix.shape[0]

       # we need to put every value of the local matrix to the global, that's why we need 2 loops: one for rows, another for columns
       for a in range(n_local):
              for b in range(n_local):
                     global_matrix[coord[a], coord[b]] += local_matrix[a, b]
       

       return global_matrix

# get global matrices
def get_global_matrices(tri_coord_sort, domain, n_nodes):

       '''
       This method obtains the global matrices by putting local matrices in the global

       :param tri_coord_sort: a list of sorted global nodes of the triangle
       :param domain: the list of all coordinates of the nodes
       :param n_nodes: the number of nodes that will be used as the rank for the global matrix
       
       :return: returns obtained global stiffness and mass matrices
       '''

       A_global = np.zeros((n_nodes, n_nodes), dtype=float)
       M_global = np.zeros((n_nodes, n_nodes), dtype=float)

       # for every triangle in the mesh
       for triangle in tri_coord_sort:
              coords = domain[triangle]

              A_local, M_local = local_triangle_matrices(coords)
              global_coords = triangle.tolist()

              put_local_to_global(A_global, A_local, global_coords)
              put_local_to_global(M_global, M_local, global_coords)

       return A_global, M_global

def get_boundary_and_interior_nodes(domain):

       '''
       This method finds the nodes that are on the boundary and that are inside of the domain

       :param domain: the (N, 2) array of all coordinates of the nodes in this domain
       :return: returns a list of the nodes that are on the boundary and list of the interior nodes
       '''

       boundary_nodes = []
       interior_nodes = []

       # get the node and its coordinates
       for i, (x, y) in enumerate(domain):
              # check if any of the coordinates are on the boundary
              if x == 0 or x == 1 or y == 0 or y == 1:
                     boundary_nodes.append(i)
              else:
                     interior_nodes.append(i)

       return boundary_nodes, interior_nodes


def apply_dirichlet(A_global, M_global, interior_nodes):

       '''
       apply_dirichlet is the method that reduces the global matrices according to the Dirichet Boundary Condition
              all the nodes on the boundary will be excluded from the global matrices because they will be zero

       :param A_global: global stiffness matrix
       :param M_global: global mass matrix
       :param interior_nodes: the list of the interior nodes

       :return: returns the reduced global stiffness and mass martices based on the interior nodes of the domain
       '''

       # reducing matrices based on boundary condition, that u = 0 on the boundary
       A_reduced = A_global[np.ix_(interior_nodes, interior_nodes)]
       M_reduced = M_global[np.ix_(interior_nodes, interior_nodes)]

       return A_reduced, M_reduced

def first_eigval_error(comp_eigval):

       '''
       This method calculates the error of the first eigenvalue

       :param comp_eigval: the first eigenvalue that was obtained numerically
       :return: returns the difference between the exact and numerical values
       '''

       real_eigval = 2 * (np.pi)**2
       error = abs(real_eigval - comp_eigval)
       
       return error

def second_third_eigval_error(comp_eigval_second, comp_eigval_third):

       '''
       This method calculates the error of the second and third eigenvalues
              because for the unit square the second and the third values are repetititve, 
                     therefore we need to compare both of them to the exact value

       :param comp_eigval_second: the second eigenvalue that was obtained numerically
       :param comp_eigval_third: the third eigenvalue that was obtained numerically

       :return: returns the difference between the exact and numerical values
       '''

       real_eigval = 5 * (np.pi)**2
       error_second = abs(real_eigval - comp_eigval_second)
       error_third = abs(real_eigval - comp_eigval_third)
       # taking the average between second and third because comparing the same real eigenvalue
       error = (error_second + error_third) / 2

       return error




In [ ]:
### VISUALISATIONS ###

def visualise_mesh(domain, triangle):

       '''
       visualise_mesh will help to see the location of the nodes and local triangles

       :param domain: the list of coordinates of all nodes
       :param triangle: the list of coordinates of the nodes of the local triangle
       '''

       plt.triplot(domain[:,0], domain[:,1], triangle.simplices.copy())
       plt.plot(domain[:,0], domain[:,1], "o")

       # to see the nodes on the graph
       # enumerate shows which node number corresponds to the coordinates
       for i, (x, y) in enumerate(domain):
              plt.text(x, y, f"P{i}", fontsize=9)

       # labeling the triangles
       for k, tri in enumerate(triangle.simplices):

              centroid = domain[tri].mean(axis=0)

              plt.text(centroid[0], centroid[1], f"T{k}", color="red", fontsize=6)

       # gca - get current axes
       plt.gca().set_title("Mesh visualisation")
       
       # set_aspect("equal") prevents stretching the plot, if it's square it will look like square
       plt.gca().set_aspect("equal")
       plt.show()

# visualise the FEM function reconstructed from the nodal values
def visualise_FEM(eigenvectors, domain, tri, n_nodes, interior_nodes):

       '''
       This method visualises the numerical functions that were found using finite element method

       :param eigenvectors: the coordinates of the eigenvector that were obtained numerically
       :param domain: the list of coordinates of all nodes
       :param tri: triangulations
       :param n_nodes: number of the nodes
       :param interior_nodes: the list of the interior nodes (that are not on the boundary)
       '''

       # after the 3rd the graphs become unnecessary because of the lack of precision
       for k in range(min(3, eigenvectors.shape[1])):
              # get k-th eigenvector of the global reduced system
              v_k = eigenvectors[:, k]

              u = np.zeros(n_nodes)
              u[interior_nodes] = v_k # use interior nodes, because boundary nodes will vanish due to Dirichet boundary condition


              plt.tripcolor(
                     domain[:, 0], # x axis
                     domain[:, 1], # y axis
                     tri.simplices, # will give the coordinates of the triangles
                     u,
                     shading="gouraud" # will give the smooth graph, where colours will change gradually
                     # can also use flat
              )

              # will help to see the difference in heights of the eigenfunctions
              plt.colorbar()
              plt.gca().set_aspect("equal")
              plt.gca().set_title(f"Eigenfunction with {k+1}-th eigenvector")
              
              plt.show()

              

def visualise_convergence(dofs, errors):

       '''
       This method plots the convergence of the error against the degrees of freedom.
              In Finite Element method the number of degrees of freedom (number of the interior nodes in the mesh)
              determines the accuracy of the approximation - a finer mesh means more nodes, 
              which reduces the error
              
       :param dofs: a list of degrees of freedom (interior nodes)
       :param errors: a list of errors
       '''

       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(dofs, errors, marker="o")
       ax.set_xlabel(r"Degrees of Freedom")
       ax.set_ylabel(r"$E(h) = |\lambda_1 - \lambda_1^h|$") # using LaTeX to express the equation for finding the error
       ax.set_title("Convergence of the FEM eigenvalues")
       ax.grid(True)
       plt.show()

def visualise_convergence_rate(base, hs, errors, p_global, expected_order, title):

       '''
       This method shows the graph of the convergence rate

       :param base: the base of the logarithm which was calculated by the ratios between the neighbouring results
       :param h: a list of the steps between the nodes, calculated as h = 1 / (n_nodes - 1)
       :param errors: a list of errors 
       '''
       hs = np.array(hs)
       errors = np.array(errors)

       # some of the errors can be nan 
       finite_error = np.isfinite(errors)

       if not np.any(finite_error):
              raise ValueError("No finite errors available to plot")
       
       # so we find the first finite value
       first_finite = np.where(finite_error)[0][0]
       
       theor = errors[first_finite] * (hs / hs[first_finite])**expected_order

       fig, ax = plt.subplots(figsize=(10, 8))
       # using the log with a special base because ratio between number of points may differ
       ax.plot(np.emath.logn(base, hs), np.emath.logn(base, errors), marker="o", label=f"Observed conv rate = {p_global:.5f}")
       ax.plot(np.emath.logn(base, hs), np.emath.logn(base, theor), "--", label=f"Theoretical conv rate = {expected_order:.5f}")
       
       ax.set_xlabel(f"log_{base:.3f}(h)")
       ax.set_ylabel(f"log_{base:.3f} (E(h))")
       ax.set_title(f"Convergence rate of the {title}")
       ax.grid(True)
       ax.legend()
       plt.show()

def visualise_eigenfunctions(discrete_eigf, exact_eigf, interior_nodes):

       '''
       Shows the comparison between numerically obtained eigenfunction and exact

       :param discrete_eigf: the eigenfunction that was computed numerically
       :param exact_eigf: the exact eigenfunction
       :param interior_nodes: the list of the interior nodes
       '''
       
       # reshape eigenvectors into square arrays

       # boundary nodes will vanish at the boundary, so we need to know the length of the side of the square produced by interior nodes
       interior_dim = int(np.sqrt(len(interior_nodes))) # int to prevent it being a float, because it won't be accepted when the eigenfunction matrix will be reshaped
       discrete_eigf = discrete_eigf.reshape(interior_dim, interior_dim)
       exact_eigf = exact_eigf.reshape(interior_dim, interior_dim)
       
       fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
       fig.suptitle("Numerical Eigenfunction & Exact Eigenfunction")
       
       ax1.imshow(discrete_eigf)
       ax1.set_title("FEM")

       ax2.imshow(exact_eigf)
       ax2.set_title("Exact")

       plt.show()
       

In [ ]:
### CHECKS FOR MATRIX SYMMETRY AND ROW SUM CHECK ###
# for finite element method the stiffness and mass matrices should be symmetric
# and row sum should be equal to 0

def sanity_check(A_global, M_global):
       print("\nCHECKS FOR MATRIX SYMMETRY AND ROW SUM CHECK\n")
       print(f"Is the global stiffness matrix symmetric? {np.allclose(A_global, A_global.T)}")
       print(f"Is the global mass matrix symmetric? {np.allclose(M_global, M_global.T)}")

       # may have floating point error 
       print(f"Is the row sum of the global stiffness matrix is 0? {np.allclose(A_global.sum(axis=1), 0)}\n")


In [ ]:
### CONVERGENCE STUDY ###

# find local convergence rate
def local_convergence_rate(row_num, n_points_list, errors):
       
       '''
       This method finds the local convergence rate (between two neighbouring rates), 
       which in other words means "how fast are we approaching the exact value"

       :param row_num: detects where are we in the table of convergence
       :param n_points_list: list of numbers of points, 
       we will need them to find the ratio between the number of points because it will determine the convergence rate
       :param errors: a list of errors that we got from comparing the exact value with numerical

       :return: returns the local rate of convergence or
       None if it is the first row or if the error is 0 or negative to ensure that we won't face division by nothing or zero
       '''

       if row_num == 0:
              
              return np.nan
              
       # prevents the cases when the error is 0
       e_old = errors[row_num-1]
       e_new = errors[row_num]
       if e_old <= 0 or e_new <= 0:
              return np.nan

       # find the local convergence rate

       # we need to find the base for the logarithm first
       ratio_between_num_points = n_points_list[row_num] / n_points_list[row_num-1] 
       # find the ratio between 2 neighbouring local errors
       ratio_between_errors = errors[row_num-1] / errors[row_num]
       # calculate the local rate of convergence
       p_local = math.log(ratio_between_errors, ratio_between_num_points)

       return p_local



In [ ]:
### COMPARE NUMERICAL AND EXACT EIGENFUNCTIONS ###

# we need to evaluate it at every node and compare with first eigenvector
def compare_numerical_with_exact(node_coords, u_h, M, A):

       '''
       This method compares the first eigenfunction with the first exact eigenfunction which for the unit square is
       u(x,y) = sin(pi * x) * sin(pi * y)

       :param node_coords: the (N, 2) array of the (x, y) coordinates of the nodes of the domain
       :param u_h: the numerically computed eigenfunction
       :param M: mass matrix

       :return: returns 2 values: first is the list of exact values of the function at each node
       second is the numerical eigenfunction
       '''
       node_coords = np.asarray(node_coords)
       x, y = node_coords[:, 0], node_coords[:, 1]
       u_exact = np.sin(np.pi * x) * np.sin(np.pi * y)

       # handle the sign (because eigenfucntions are only determined up to a sign), 
       # so two vectors that represent the same function pointing in opposite direction (dot product is < 0) this will give a huge error when calculating the norm
       if np.dot(u_exact, u_h) < 0:
              u_h = -u_h

       # normalise to work in the same ratios
       u_exact /= np.sqrt(u_exact @ M @ u_exact)
       u_h /= np.sqrt(u_h @ M @ u_h)

       l2_error = np.sqrt((u_exact - u_h) @ M @ (u_exact - u_h))

       # H1 error
       w = u_exact - u_h
       h1_error = np.sqrt(w.T @ (M + A) @ w)
       
       return u_exact, u_h, l2_error, h1_error

def compare_second_third_eigf_with_exact(node_coords, u_h_2, u_h_3, M, A):

       '''
       This method compares the second and third eigenfunction with the second and third exact eigenfunction which for the unit square is
       u(x,y) = sin(2 * pi * x) * sin(pi * y) or u(x,y) = sin(pi * x) * sin(2 * pi * y)

       :param node_coords: the (N, 2) array of the (x, y) coordinates of the nodes of the domain
       :param u_h_2: the numerically computed second eigenfunction
       :param u_h_3: the numerically computed third eigenfunction

       :return: returns 3 values: first is the list of exact values of the function at each node
       second and third are the numerical eigenfunctions
       
       '''
       node_coords = np.asarray(node_coords)
       x, y = node_coords[:, 0], node_coords[:, 1]

       exact_12 = np.sin(np.pi * x) * np.sin(2 * np.pi * y)
       exact_21 = np.sin(2 * np.pi * x) * np.sin(np.pi * y)

       exact_12 /= np.sqrt(exact_12 @ M @ exact_12)
       exact_21 /= np.sqrt(exact_21 @ M @ exact_21)
       # sanity check: exact_12 @ M @ exact_21 should be approx 0

       def project(v):
              c1 = v @ M @ exact_12
              c2 = v @ M @ exact_21
              return c1 * exact_12 + c2 * exact_21
       
       Pe_2, Pe_3 = project(u_h_2), project(u_h_3)
       proj_error_2 = np.sqrt((u_h_2 - Pe_2) @ M @ (u_h_2 - Pe_2))
       proj_error_3 = np.sqrt((u_h_3 - Pe_3) @ M @ (u_h_3 - Pe_3))

       w2 = u_h_2 - Pe_2
       w3 = u_h_3 - Pe_3

       h1_error_2 = np.sqrt(w2.T @ (M + A) @ w2)
       h1_error_3 = np.sqrt(w3.T @ (M + A) @ w3) 

       return exact_12, exact_21, Pe_2, Pe_3, proj_error_2, proj_error_3, h1_error_2, h1_error_3

In [ ]:
### THE FULL FEM ANALYSIS ###

def run_FEM_analysis(n_points, n_points_list, 
                     errors, errors_second_third, errors_eigf_l2, errors_eigf_proj_2, errors_eigf_proj_3,
                     errors_eigf_h1, errors_eigf_h1_2, errors_eigf_h1_3, 
                     index):

       '''
       This method combines all the methods to perform finite element method computations for the unit square

       :param n_points: the number of points per axis in the unit square
       :param n_points_list: the list of number of points per axis in the unit square
       :param errors: the list of errors between the exact and numerical values
       :param errors_second_third: the list of errors between the exact eigenvalue and the second and third
       (the exact eigenvalue would repeat itself, that's why we compare with second and third)
       :param errors_eigf: the list of errors between numerical eigenfunctions and exact
       :param index: shows what round of the loop we are having (will need for tables)

       :return: returns a dictionary that stores:
       h - the step between the nodes
       n_points - the number of points per axis
       dofs - list of degrees of freedom (the number of the interior nodes)
       eigval - the first eigenvalue
       error_eigval - the error of the first eigenvalue
       error_23 - the error of the second and third eigenvalues against the exact eigenvalue
       error_eigf - the error of the approximation of the exact eigenfunction
       p_eigval - the convergence rate of the eigenvalues (how fast they approach the exact value)
       p_eigf - the convergence rate of the eigenfunctions (how fast do the approximations approach the function)
       second_eigvec_second_eigf - the dot product of the second eigenvector and second exact eigenfunction
       second_eigvec_third_eigf - the dot product of the second eigenvector and third exact eigenfunction
       third_eigvec_second_eigf - the dot product of the third eigenvector and second exact eigenfunction
       third_eigvec_third_eigf - the dot product of the third eigenvector and third exact eigenfunction
       '''
       
       ### Mesh Generation

       # generate mesh and split it into triangles
       domain, triangle = generate_mesh(n_points)
       # sort nodes of each triangle into ascending order so they are consistently ordered
       tri_coord_sort = np.sort(triangle.simplices)
       # optional visualisation for small meshes
       if n_points < 15: # everything above 15 becomes indistinguishable on the plot
              visualise_mesh(domain, triangle)


       ### Getting Global Matrices

       n_nodes = len(domain)
       A_global, M_global = get_global_matrices(tri_coord_sort, domain, n_nodes)
       # check if the matrices are symmetric and if the row sum is 0
       sanity_check(A_global, M_global)

       

       ### Boundary conditions

       boundary_nodes, interior_nodes = get_boundary_and_interior_nodes(domain)
       # apply dirichlet BC
       A_reduced, M_reduced = apply_dirichlet(A_global, M_global, interior_nodes)

       
       ### Eigenvalues

       # finding eigenvalues and its errors
       # SciPy stores eigvectors as columns
       eigvals, eigvecs = eigh(A_reduced, M_reduced)

       print(f"The eigenvalues: \n{eigvals[0]}\n")

       # eigenvalue error
       error = first_eigval_error(eigvals[0])
       errors.append(error)
       if n_points < 4: # prevents the error, because with 3 points per axis there is only one eigenvalue
              errors_second_third.append(np.nan)
              error_23 = np.nan
              eigval_2 = np.nan
              eigval_3 = np.nan
              
       else:
              eigval_2, eigval_3 = eigvals[1], eigvals[2]
              error_23 = second_third_eigval_error(eigvals[1], eigvals[2])
              errors_second_third.append(error_23)

       
       ### Eigenfunctions

       # eigenfunction comparison
       interior_nodes_coords = [domain[interior_node] for interior_node in interior_nodes] 
       u_exact, u_h, l2_error, h1_error = compare_numerical_with_exact(interior_nodes_coords, eigvecs[:,0], M_reduced, A_reduced)
       errors_eigf_l2.append(l2_error)
       errors_eigf_h1.append(h1_error)
       # compare second and third with the exact
       if n_points < 4: # for 3 points we won't get the second and third eigenvectors
              errors_eigf_proj_2.append(np.nan)
              errors_eigf_proj_3.append(np.nan)
              exact_eigf_12 = exact_eigf_21 = None
              proj_error_2 = proj_error_3 = np.nan
              errors_eigf_h1_2.append(np.nan)
              errors_eigf_h1_3.append(np.nan)
              h1_error_2 = h1_error_3 = np.nan
       else:
              exact_eigf_12, exact_eigf_21, proj_2, proj_3, proj_error_2, proj_error_3, h1_error_2, h1_error_3 = compare_second_third_eigf_with_exact(interior_nodes_coords, eigvecs[:,1], eigvecs[:, 2], M_reduced, A_reduced)
              errors_eigf_proj_2.append(proj_error_2)
              errors_eigf_proj_3.append(proj_error_3)
              errors_eigf_h1_2.append(h1_error_2)
              errors_eigf_h1_3.append(h1_error_3)


       
       ### Convergence

       h = 1 / (n_points - 1)
       p_eigval = local_convergence_rate(index, n_points_list, errors)
       p_eigval_23 = local_convergence_rate(index, n_points_list, errors_second_third)
       p_eigf_l2 = local_convergence_rate(index, n_points_list, errors_eigf_l2)
       p_eigf_proj_2 = local_convergence_rate(index, n_points_list, errors_eigf_proj_2)
       p_eigf_proj_3 = local_convergence_rate(index, n_points_list, errors_eigf_proj_3)
       p_eigf_h1 = local_convergence_rate(index, n_points_list, errors_eigf_h1)
       p_eigf_h1_2 = local_convergence_rate(index, n_points_list, errors_eigf_h1_2)
       p_eigf_h1_3 = local_convergence_rate(index, n_points_list, errors_eigf_h1_3)

       
       ### Visualisations

       # visualise FEM
       visualise_FEM(eigvecs, domain, triangle, n_nodes, interior_nodes)
       # visualise exact eigenfunction next to FEM eigenfunction
       visualise_eigenfunctions(u_h, u_exact, interior_nodes)
       

       result = {
              "h": h,
              "n_points": n_points,
              "dofs": len(interior_nodes),
              "eigval": eigvals[0],
              "eigval_23": np.mean([eigval_2, eigval_3]),
              "error_eigval": error,
              "error_eigval_23": error_23,
              "error_eigf_l2": np.round(l2_error, decimals=6),
              "error_eigf_h1": np.round(h1_error, decimals=6),
              "error_eigf_proj_2": proj_error_2,
              "error_eigf_proj_3": proj_error_3,
              "error_eigf_h1_2": h1_error_2,
              "error_eigf_h1_3": h1_error_3,
              "p_eigval": p_eigval,
              "p_eigval_23": p_eigval_23,
              "p_eigf_l2": p_eigf_l2,
              "p_eigf_h1": p_eigf_h1,
              "p_eigf_proj_2": p_eigf_proj_2,
              "p_eigf_proj_3": p_eigf_proj_3,
              "p_eigf_h1_2": p_eigf_h1_2,
              "p_eigf_h1_3": p_eigf_h1_3,
       }
       if exact_eigf_12 is not None:
              result.update({
                     "second_eigvec_second_eigf": abs(eigvecs[:, 1] @ M_reduced @ exact_eigf_12),
                     "second_eigvec_third_eigf": abs(eigvecs[:, 1] @ M_reduced @ exact_eigf_21),
                     "third_eigvec_second_eigf": abs(eigvecs[:, 2] @ M_reduced @ exact_eigf_12),
                     "third_eigvec_third_eigf": abs(eigvecs[:, 2] @ M_reduced @ exact_eigf_21)

       })

       return result

In [ ]:
# More complex mesh 
#######
##### MAIN #####
########

n_points_list = [3, 6, 12, 24, 48]

errors = []
errors_second_third = []
errors_eigf_l2 = []
errors_eigf_proj_2 = []
errors_eigf_proj_3 = []
errors_eigf_h1 = []
errors_eigf_h1_2 = []
errors_eigf_h1_3 = []

# TABLES
# error table
error_eigval_table = pd.DataFrame(columns=["Points per axis", "Interior DoF", "First eigenvalue", "Error"])
error_eigval_23_table = pd.DataFrame(columns=["Points per axis", "Interior DoF", "Degenerate eigenvalue", "Error"])

# convergence rate table
conv_eigval_table = pd.DataFrame(columns=["h", "lambda_1,h", "Error", "Rate of convergence"])
conv_eigval_23_table = pd.DataFrame(columns=["h", "lambda_2_3,h", "Error", "Rate of convergence"])

# eigenfunction error table
eigf_l2_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])

eigf_proj_2_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])
eigf_proj_3_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])

eigf_h1_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])

eigf_h1_2_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])
eigf_h1_3_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])

# eigenfunction convergence table
eigf_l2_conv_table = pd.DataFrame(columns=["h", "Error", "Rate of convergence"])
eigf_h1_conv_table = pd.DataFrame(columns=["h", "Error", "Rate of convergence"])

eigf_proj_23_conv_table = pd.DataFrame(columns=["h", "Error of second eigfunc", "Rate of convergence of second", "Error of third eigfunc", "Rate of convergence of third"])
eigf_h1_23_conv_table = pd.DataFrame(columns=["h", "Error of second eigfunc", "Rate of convergence of second", "Error of third eigfunc", "Rate of convergence of third"])

# the table of the dot products
dot_prod_table = pd.DataFrame(columns=["h", "2_2_dot", "3_3_dot", "2_3_dot", "3_2_dot"])

for index, n_points in enumerate(n_points_list):
       result = run_FEM_analysis(n_points, n_points_list, errors, 
                                   errors_second_third, errors_eigf_l2, errors_eigf_proj_2, errors_eigf_proj_3, 
                                   errors_eigf_h1, errors_eigf_h1_2, errors_eigf_h1_3, 
                            index)

       # put results in the table
       # Eigenvalue error
       error_eigval_table.loc[index + 1] = [result["n_points"], result["dofs"], result["eigval"], result["error_eigval"]]
       error_eigval_23_table.loc[index + 1] = [result["n_points"], result["dofs"], result["eigval_23"], result["error_eigval_23"]]
       # eigenvalue convergence
       conv_eigval_table.loc[index + 1] = [result["h"], result["eigval"], result["error_eigval"], result["p_eigval"]]
       conv_eigval_23_table.loc[index + 1] = [result["h"], result["eigval_23"], result["error_eigval_23"], result["p_eigval_23"]]
       
       # eigenfunctions error
       eigf_l2_error_table.loc[index + 1] = [result["n_points"], result["error_eigf_l2"]]
       eigf_h1_error_table.loc[index + 1] = [result["n_points"], result["error_eigf_h1"]]
       
       # eigenfunction convergence 
       eigf_l2_conv_table.loc[index + 1] = [result["h"], result["error_eigf_l2"], result["p_eigf_l2"]]
       eigf_h1_conv_table.loc[index + 1] = [result["h"], result["error_eigf_h1"], result["p_eigf_h1"]]
       
       # degenerate eigenfunction error
       eigf_proj_2_error_table.loc[index + 1] = [result["n_points"], result["error_eigf_proj_2"]]
       eigf_proj_3_error_table.loc[index + 1] = [result["n_points"], result["error_eigf_proj_3"]]


       eigf_h1_2_error_table.loc[index + 1] = [result["n_points"], result["error_eigf_h1_2"]]
       eigf_h1_3_error_table.loc[index + 1] = [result["n_points"], result["error_eigf_h1_3"]]
       
       # degenerate eigenfunction convergence
       eigf_proj_23_conv_table.loc[index + 1] = [result["h"], result["error_eigf_proj_2"], result["p_eigf_proj_2"], result["error_eigf_proj_3"], result["p_eigf_proj_3"]]
       eigf_h1_23_conv_table.loc[index + 1] = [result["h"], result["error_eigf_h1_2"], result["p_eigf_h1_2"], result["error_eigf_h1_3"], result["p_eigf_h1_3"]]
       
       
       if index >= 1:
              dot_prod_table.loc[index] = [result["h"], result["second_eigvec_second_eigf"], result["third_eigvec_third_eigf"], result["second_eigvec_third_eigf"], result["third_eigvec_second_eigf"]]


# Error table
print(f"\nFirst eigenvalue ERROR DEPENDENCY ON THE NUMBER POINTS PER AXIS\n{error_eigval_table}")
print(f"\nSecond and third eigenvalue ERROR DEPENDENCY ON THE NUMBER POINTS PER AXIS\n{error_eigval_23_table}")

# Convergence rate
print(f"\nFirst eigenvalue CONVERGENCE RATE TABLE\n {conv_eigval_table}")
print(f"\nSecond and third eigenvalue CONVERGENCE RATE TABLE\n {conv_eigval_23_table}")

# Error between the exact and numerical eigenfunctions
print(f"\nL2 ERROR BETWEEN EXACT AND NUMERICAL EIGENFUNCTIONS\n{eigf_l2_error_table}")
print(f"\nH1 ERROR BETWEEN EXACT AND NUMERICAL EIGENFUNCTIONS\n{eigf_h1_error_table}")

print(f"\nProjection ERROR BETWEEN second EXACT AND NUMERICAL EIGENFUNCTIONS\n{eigf_proj_2_error_table}")
print(f"\nProjection ERROR BETWEEN third EXACT AND NUMERICAL EIGENFUNCTIONS\n{eigf_proj_3_error_table}")


# convergence rate of the eigenfucntions
print(f"\nEIGENFUNCTION L2 CONVERGENCE RATE TABLE\n{eigf_l2_conv_table}\n")
print(f"\nEIGENFUNCTION H1 CONVERGENCE RATE TABLE\n{eigf_h1_conv_table}\n")
print(f"\nDegenerate EIGENFUNCTION Projection CONVERGENCE RATE TABLE\n{eigf_proj_23_conv_table}\n")
print(f"\nDegenerate EIGENFUNCTION H1 CONVERGENCE RATE TABLE\n{eigf_h1_23_conv_table}\n")

# dot product table
print(f"\nDOT PRODUCT TABLE OF EIGENVECTOR AND EXACT EIGENFUNCTION\n{dot_prod_table}")


# Plots
dofs = error_eigval_table["Interior DoF"].tolist()
hs = conv_eigval_table["h"].tolist()

ratio_between_num_points = [n_points_list[i] / n_points_list[i-1] for i in range(1, len(n_points_list))] 
base = np.mean(ratio_between_num_points)


p_global_eigval = np.nanmean(conv_eigval_table["Rate of convergence"].tolist()[2:])
p_global_eigval_23 = np.nanmean(conv_eigval_23_table["Rate of convergence"].tolist()[2:])

p_global_eigf_l2 = np.nanmean(eigf_l2_conv_table["Rate of convergence"].tolist()[2:])
p_global_eigf_h1 = np.nanmean(eigf_h1_conv_table["Rate of convergence"].tolist()[2:])

p_global_eigf_proj_2 = np.nanmean(eigf_proj_23_conv_table["Rate of convergence of second"].tolist()[2:])
p_global_eigf_proj_3 = np.nanmean(eigf_proj_23_conv_table["Rate of convergence of third"].tolist()[2:])

p_global_eigf_h1_2 = np.nanmean(eigf_h1_23_conv_table["Rate of convergence of second"].tolist()[2:])
p_global_eigf_h1_3 = np.nanmean(eigf_h1_23_conv_table["Rate of convergence of third"].tolist()[2:])


print(f"\nThe convergence rate of the eigenvalues is {p_global_eigval:.5f}")
print(f"\nThe convergence rate of L2 approximated eigenfunctions is {p_global_eigf_l2:.5f}")
print(f"\nThe convergence rate of H1 approximated eigenfunctions is {p_global_eigf_h1:.5f}")

# log-log plot and fit a line
p_polyfit_eigval = np.polyfit(np.emath.logn(base, hs), np.emath.logn(base, errors), 1)[0]
print(f"Observed rate = {p_polyfit_eigval:.5f}")
       
# plotting convergence rate of eigenvalues
visualise_convergence_rate(base, hs, errors, p_global_eigval, 2, "first eigenvalue")
visualise_convergence_rate(base, hs, errors_second_third, p_global_eigval_23, 2, "degenerate eigenvalue")


# Eigefunctions
# L2
visualise_convergence_rate(base, hs, errors_eigf_l2, p_global_eigf_l2, 2, "L2 error of first eigenfunction")
# H1
visualise_convergence_rate(base, hs, errors_eigf_h1, p_global_eigf_h1, 1, "H1 error of first eigenfunction")

# Degenerate
# projection
visualise_convergence_rate(base, hs, np.nanmean([errors_eigf_proj_2, errors_eigf_proj_3], axis=0), np.nanmean([p_global_eigf_proj_2, p_global_eigf_proj_3]), 2, "L2 projection error of degenerate eigenfunction")
#visualise_convergence_rate(base, hs, errors_eigf_proj_3, p_global_eigf_proj_3, 2, "projection error of third eigenfunction")

visualise_convergence_rate(base, hs, np.nanmean([errors_eigf_h1_2, errors_eigf_h1_3], axis=0), np.nanmean([p_global_eigf_h1_2, p_global_eigf_h1_3]), 1, "H1 projectionerror of degenerate eigenfunction")
#visualise_convergence_rate(base, hs, errors_eigf_h1_3, p_global_eigf_h1_3, 1, "H1 error of third eigenfunction")



**RESULTS AND OBSEVATIONS**
- $\lambda_1$ decreases with the better mesh.
- the error decreases because the finite-dimentional subspace grows and can represent smoother functions.
- The eigenfunction has a shape of a smooth hill.
- Writing u(x,y) = X(x)Y(y) splits the Laplacian eigenvalue problem into two independent 1D problems, each giving sine solutions. The lowest-frequency solution in each direction is sin(πx) and sin(πy), so the first eigenfunction is exactly their product. FEM solution is a piecewise linear approximation to this — visually indistinguishable on fine meshes.
- if v is an eigenvector, then -v also will satisfy the eigenvalue problem. the correction is made by checking the dot product: if the dot product is negative, the vectors point in opposite directions, so the sign of u_h is flipped before computing the error. This ensures the measurement of the true approximation error.
- The second and third exact eigenfunctions correspond to the repeated eigenvalue $5\pi^2$. Since this eigenspace is two-dimensional, the finite element solver is free to return any orthonormal basis spanning the same space. Consequently, the numerical second and third eigenvectors need not match the exact eigenfunctions individually. Their projections onto the exact eigenfunctions were therefore computed using dot products.
- From the output the convergence rate of the eigenvalues is 2.173 which is approaching 2, which agrees with theory. The convergence rate of the eigenfunction approximation is 2.197 which also approaches 2, which agrees with the theory.
- Second and third eigenvectors do not match exact eigenfunctions individually.
- The solver returns an arbitrary orthonormal basis of the eigenspace
- Because the second eigenvalue is repeated, the numerical eigensolver may return any orthonormal basis of the two-dimensional eigenspace. Therefore the diagonal dot products are not expected to converge to 1 and the off-diagonal dot products are not expected to converge to 0. The observed values indicate that the numerical eigenvectors are rotated combinations of the exact eigenfunctions, which is the expected behaviour for a degenerate eigenspace.

**Degenerate (repeated) eigenspace**

- the second and third eigenvalues are repeated because of the formula which is $\lambda_1,2 = \pi^2 (m^2 + n^2)$ with exact eigenfunctions $u_m,n (x, y) = sin(m*\pi *x)sin(n*\pi *y)$. So when we try to find the eigenvalues we get $\lambda_1,2 = \pi^2 (1^2 + 2^2) = 5\pi^2 = \lambda_2,1$. This is called eigenspace of dimention 2
- in addition, we can look at algebraic multiplicity, which will be 2 (the root appears twice), while the geometric multiplicity is also 2 (the number of linearly independent eigenfunctions for that eigenvalue: we have $sin(2*\pi *x)sin(\pi *y)$ and $sin(\pi *x)sin(2*\pi *y)$). Therefore we can say that two eigenfunctions span a 2D eigenspace and any linear combination of them is also a valid eigenfucntion

The numerical solver is not returning the $u_1,2 or u_2,1$ - it is returning some arbitrary rotation of the 2D eigenspace basis
- the dot products are not close to 1, because the numerical eigenvectors are a rotation of the exact basis by some angle $\theta$, the dot product gives the $cos(\theta)$ and $sin(\theta)$. If we look at the table we will see that the (2_2_dot)^2 + (2_3_dot)^2 $\approx 0.6^2 + 0.8^2 = 1$. This is Pythagoras theorem, i.e. the eigenspace is perfectly recovered, just angled. 
- The jump to 0.84 in the last row can be explained that on the finer meshes, numerical roundoff breaks teh near-repetence differently, making a solver to use different rotation
- (better explanation) the eigensolver is free to choose any orthonormal basis of the approximate 2D eigenspace. Small changes in the discretisation may therefore lead to different rotations of the numerical basis, even though the eigenspace itself is converging correctly. That's why convergence of individual eigenvectors is not an appropriate measure for repeated eigenvalue. Instead the convergence should be assessed at the level of the eigenspace itself

- The better thing would be to check the convergence of the error between eigenvector and its projection (or in other words check if the 2D eigenspace is recovered correctly) Compute:  $P_e v_k = ⟨v_k, u_1,2⟩ * u_1,2 + ⟨v_k, u_2,1⟩ * u_2,1$
Then check  ‖vₖ − Pₑ vₖ‖  converges to 0 as h → 0. Where P_e projects onto the span ${u_1,2 , u_2,1}


**Why the P1 uses matrices and P2 quadrature**

Q2 — yes, they're genuinely different quantities, not two implementations of the same formula, and you should say so explicitly rather than treat it as a footnote. [Likely, standard FEM argument] The quadrature approach in your P2 code evaluates u_exact directly at quadrature points and compares against u_h, it approximates the true error against the smooth exact solution. The P1 matrix trick, w @ (M+A) @ w, only ever sees nodal values, so it exactly computes the H1 norm of the difference between the piecewise-linear interpolant of u and u_h, not u itself
Should you worry it breaks your results? No. [Likely] By the triangle inequality, and interpolation error ||u - I_h u||H1 for P1 is itself O(h) — same order as the FEM discretization error you're trying to measure. So the two quantities are asymptotically equivalent in convergence rate (bothO(h)), even though their absolute numerical values at a given h won't match exactly. What you should not do is present the two side by side implying they're the same measurement — a sentence like "the P1 H1 error is computed algebraically via w^T(M+A)w using nodal interpolation, whereas P2 uses direct quadrature evaluation against the exact solution; both target the same O(h^k) asymptotic rate but are not numerically identical quantities" is the honest framing, and it's a legitimate methodological choice